# Generate public-safe dissertation figures (aggregate)

This notebook regenerates **public-safe aggregate** dissertation figures from the cleaned GitHub repository.

## Purpose
- Produce presentation-quality **aggregate** figures for Figures 1–3.
- Avoid any use of protected participant-level audio, annotations, or clinical labels.

## Required inputs (public-safe)
- `results/final_matched_metrics_table.csv`

## Privacy note
Raw clinical audio, participant-level labels, and private annotation exports are **not included** in this repository.

## What can be regenerated here
- **Figure 1 (cohort flow):** regenerated from **aggregate counts** (hard-coded below).
- **Figure 2 (coverage + AF/SR-classified performance):** regenerated from `final_matched_metrics_table.csv`.
- **Figure 3 (PCG confusion matrix):** regenerated from **aggregate confusion-matrix counts** (hard-coded below).

## What is excluded from the public repository
The following are derived from participant-level signal data and are excluded from public GitHub:
- Figure 4 (RR tachogram / Lorenz examples)
- Figure M1 (raw vs preprocessed waveform example)
- Figure M2 (S1/S2 annotation example)

If you have access to the protected dataset, those figures can be regenerated locally in a secure environment.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_context("talk")
sns.set_style("whitegrid")

REPO_ROOT = Path(".").resolve()
RESULTS_CSV = REPO_ROOT / "results" / "final_matched_metrics_table.csv"
OUT_DIR = REPO_ROOT / "figures" / "main" / "regenerated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Using:", RESULTS_CSV)
print("Writing to:", OUT_DIR)


## Load aggregate metrics table

This table contains **aggregate** coverage and AF/SR-classified diagnostic metrics for each method on the matched-to-PCG cohort (`n_ref=67`).


In [ ]:
df = pd.read_csv(RESULTS_CSV)

display(df)

assert set(df["n_ref"].tolist()) == {67}, "Expected matched cohort denominator n_ref=67 for all methods"


## Figure 1 (regenerated): Cohort flow (aggregate counts)

These are **aggregate counts** used to render a cohort-flow diagram.

Note: In the dissertation, the final version may use a different visual style, but the numbers should match the reported cohort accounting.


In [ ]:
# Aggregate cohort flow counts (public-safe)
flow = {
    "Total in device label table": 182,
    "With PCG WAV": 159,
    "With ECG12 AF/SR label": 68,
    "ECG12 AF/SR with PCG available": 67,
    "PCG AF/SR-classified": 26,
    "PCG uninterpretable (UI)": 41,
}

flow_class_breakdown = {
    "PCG AF/SR-classified": {"AF": 8, "SR": 18},
    "PCG uninterpretable (UI)": {"AF": 9, "SR": 32},
}

decided_rate = flow["PCG AF/SR-classified"] / flow["ECG12 AF/SR with PCG available"]
ui_rate = flow["PCG uninterpretable (UI)"] / flow["ECG12 AF/SR with PCG available"]

(decided_rate, ui_rate)


In [ ]:
# Simple flowchart using matplotlib annotations
fig, ax = plt.subplots(figsize=(10, 7))
ax.axis("off")

box = dict(boxstyle="round,pad=0.5", fc="white", ec="#333333", lw=1.5)
arrow = dict(arrowstyle="-|>", lw=1.5, color="#333333")

x = 0.05
w = 0.9

ys = [0.85, 0.68, 0.51, 0.34]
labels = [
    f"Total participants
N = {flow['Total in device label table']}",
    f"Participants with PCG recording
N = {flow['With PCG WAV']}",
    f"Participants with ECG12 AF/SR label
N = {flow['With ECG12 AF/SR label']}",
    f"Matched cohort (ECG12 AF/SR + PCG available)
N = {flow['ECG12 AF/SR with PCG available']}",
]

for y, lab in zip(ys, labels):
    ax.text(0.5, y, lab, ha="center", va="center", bbox=box)

for y0, y1 in zip(ys[:-1], ys[1:]):
    ax.annotate("", xy=(0.5, y1 + 0.07), xytext=(0.5, y0 - 0.07), arrowprops=arrow)

# Final split
left_x, right_x = 0.27, 0.73
final_y = 0.12

lab_left = (
    "PCG AF/SR-classified
"
    f"N = {flow['PCG AF/SR-classified']} ({decided_rate*100:.1f}%)
"
    f"AF = {flow_class_breakdown['PCG AF/SR-classified']['AF']}, "
    f"SR = {flow_class_breakdown['PCG AF/SR-classified']['SR']}"
)
lab_right = (
    "PCG uninterpretable (UI)
"
    f"N = {flow['PCG uninterpretable (UI)']} ({ui_rate*100:.1f}%)
"
    f"AF = {flow_class_breakdown['PCG uninterpretable (UI)']['AF']}, "
    f"SR = {flow_class_breakdown['PCG uninterpretable (UI)']['SR']}"
)

ax.text(left_x, final_y, lab_left, ha="center", va="center", bbox=box)
ax.text(right_x, final_y, lab_right, ha="center", va="center", bbox=box)

# Split arrows from matched cohort
ax.annotate("", xy=(left_x, final_y + 0.07), xytext=(0.5, ys[-1] - 0.07), arrowprops=arrow)
ax.annotate("", xy=(right_x, final_y + 0.07), xytext=(0.5, ys[-1] - 0.07), arrowprops=arrow)

fig.suptitle("Cohort flow for PCG AF/SR evaluation (aggregate)", y=0.98)
fig.tight_layout()

out_png = OUT_DIR / "fig1_cohort_flow_regenerated.png"
out_pdf = OUT_DIR / "fig1_cohort_flow_regenerated.pdf"
fig.savefig(out_png, dpi=300)
fig.savefig(out_pdf)
plt.close(fig)

print("Wrote:", out_png)
print("Wrote:", out_pdf)


## Figure 2 (regenerated): Coverage and AF/SR-classified performance

Panel A uses the matched cohort denominator (`n_ref=67`) and shows the proportion of outputs that are:
- AF/SR-classified
- OA (other arrhythmia / indeterminate)
- UI (uninterpretable)
- Missing

Panel B reports sensitivity, specificity, and accuracy **computed only among AF/SR-classified outputs**.


In [ ]:
order = ["PCG", "FibriCheck iOS", "FibriCheck Android", "Kardia"]
df2 = df.set_index("method").loc[order].reset_index()

# Coverage proportions
coverage = pd.DataFrame({
    "method": df2["method"],
    "AF/SR-classified": df2["classified_rate"],
    "OA": df2["OA_rate"],
    "UI": df2["UI_rate"],
    "Missing": df2["missing_rate"],
}).set_index("method")

# Performance metrics on AF/SR-classified subset
perf = pd.DataFrame({
    "method": df2["method"],
    "Sensitivity": df2["sensitivity"],
    "Specificity": df2["specificity"],
    "Accuracy": df2["accuracy"],
}).set_index("method")

coverage, perf


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: stacked coverage
ax = axes[0]
left = np.zeros(len(coverage))
colors = {
    "AF/SR-classified": "#4C78A8",
    "OA": "#F58518",
    "UI": "#B8B8B8",
    "Missing": "#E45756",
}
for col in ["AF/SR-classified", "OA", "UI", "Missing"]:
    vals = coverage[col].to_numpy(dtype=float)
    ax.bar(coverage.index, vals, bottom=left, label=col, color=colors[col], edgecolor="white")
    left += vals

ax.set_ylim(0, 1)
ax.set_ylabel("Proportion of matched cohort")
ax.set_title("A. Coverage")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)

# annotate classified counts on bars
for i, method in enumerate(df2["method"].tolist()):
    n_class = int(df2.loc[i, "n_classified_AF_SR"])
    n_ref = int(df2.loc[i, "n_ref"])
    pct = 100.0 * float(n_class) / float(n_ref)
    ax.text(i, 1.02, f"{n_class}/{n_ref}
classified
({pct:.1f}%)", ha="center", va="bottom", fontsize=10)

# Panel B: performance
ax = axes[1]
perf_plot = perf.reset_index().melt(id_vars="method", var_name="metric", value_name="value")
sns.barplot(data=perf_plot, x="method", y="value", hue="metric", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Metric value")
ax.set_title("B. AF/SR-classified performance")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)

# x tick formatting
ax.set_xticklabels([
    "PCG
(n=26)",
    "FibriCheck iOS
(n=61)",
    "FibriCheck Android
(n=38)",
    "Kardia
(n=57)",
])
axes[0].set_xticklabels([
    "PCG", "FibriCheck iOS", "FibriCheck Android", "Kardia"
], rotation=0)

fig.suptitle("Coverage and AF/SR-classified performance (matched cohort)", y=1.02)
fig.tight_layout()

out_png = OUT_DIR / "fig2_coverage_performance_regenerated.png"
out_pdf = OUT_DIR / "fig2_coverage_performance_regenerated.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)

print("Wrote:", out_png)
print("Wrote:", out_pdf)


## Figure 3 (regenerated): PCG confusion matrix (aggregate counts)

The public repository does **not** include participant-level probabilities, so the **ROC curve cannot be regenerated** exactly from aggregate tables alone.

However, the confusion matrix for the AF/SR-classified subset can be shown using aggregate counts.


In [ ]:
# Aggregate confusion matrix counts for PCG on the AF/SR-classified subset (public-safe)
# Rows: ECG12 reference (SR, AF); Columns: PCG prediction (SR, AF)
#
# True SR predicted SR: 16
# True SR predicted AF: 2
# True AF predicted SR: 1
# True AF predicted AF: 7
cm = np.array([[16, 2], [1, 7]], dtype=int)
cm


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    linewidths=0.0,
    ax=ax,
)

ax.set_xlabel("Predicted label")
ax.set_ylabel("ECG12 reference")
ax.set_xticklabels(["SR", "AF"])
ax.set_yticklabels(["SR", "AF"], rotation=0)
ax.set_title("PCG confusion matrix (AF/SR-classified subset)")

out_png = OUT_DIR / "fig3_pcg_confusion_matrix_regenerated.png"
out_pdf = OUT_DIR / "fig3_pcg_confusion_matrix_regenerated.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)

print("Wrote:", out_png)
print("Wrote:", out_pdf)
